## Generate Embeddings and Store in Vector Database (FAISS)

In this section, we convert each chunk of OCR-extracted text into numerical vectors using a pre-trained transformer model. These vectors are then stored in a FAISS index for efficient similarity search. Metadata is also saved for mapping results back to text.

This step prepares the text chunks for semantic search by converting them into vector representations using a transformer model. The resulting vectors are stored in a FAISS index, which allows efficient similarity-based retrieval. Metadata is also saved to maintain a link between vectors and their source content.

**Step 1: Imported requiered libraries** 

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import os
import numpy as np

c:\Users\brian\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Step 2: Load the Chunked Text File**

We load the chunked text file generated from PaddleOCR. Each row contains a small portion of text (chunk), usually ~500 tokens, and associated metadata such as filename and chunk ID.

In [2]:
#Load the chunked PaddleOCR CSV file
csv_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\extracted_text_PaddleOCR2\cleaned_text_PaddleOCR2\chunked_text_PaddleOCR.csv"
df = pd.read_csv(csv_path)
df.head(n=3)

,chunk_id,filename,chunk_text
0,0_0.txt_0,0_0.txt,[ WIKIPEDIA Donate Create account Log in . The...
1,0_1.txt_1,0_1.txt,[ Animation [ edit ] Year Title Role Notes 199...
2,0_10.txt_2,0_10.txt,"[ This page was last edited on 20 April 2025 ,..."


**Step 3: Load the Sentence Embedding Model**

We use sentence-transformers to convert each text chunk into a dense vector. The model "all-MiniLM-L6-v2" provides a balance between accuracy and speed.

In [3]:
# This model converts text into vector form
model = SentenceTransformer("all-MiniLM-L6-v2")

**Step 4: Generate Embeddings for All Chunks**

We extract the column "chunk_text" and pass each chunk to the embedding model. The result is a high-dimensional vector representation for each chunk. FAISS requires float32 format, so we convert the array.


In [4]:
# Generate embeddings for all chunks 
# Get list of text chunks
texts = df["chunk_text"].tolist()  
# Get sentence embeddings
embeddings = model.encode(texts, show_progress_bar=True)  
embeddings = np.array(embeddings).astype("float32")  

Batches: 100%|██████████| 454/454 [17:26<00:00,  2.30s/it]


**Step 5: Build and Populate the FAISS Index**

FAISS is a vector search library optimized for speed. We use FlatL2 index type (brute-force with Euclidean distance). This structure will let us perform fast similarity queries later.

In [6]:
# Get the size of each embedding vector
dimension = embeddings.shape[1]  
# Create a FAISS index using L2 (Euclidean) distance
index = faiss.IndexFlatL2(dimension)  
# Add all embeddings to the index
index.add(embeddings)  

**Step 6: Save the Index and Metadata**

We store the vector index for reuse, and also save the corresponding metadata. The metadata contains filenames and chunk text that allow us to interpret search results returned by FAISS later.

In [7]:
# Save the FAISS index and metadata ===
# Save the index file
faiss_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\faiss_index_PaddleOCR.index"
faiss.write_index(index, faiss_path)

# Save the metadata (chunk_id, filename, chunk_text) for later use
metadata_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\metadata_PaddleOCR.csv"
df.to_csv(metadata_path, index=False)